# OSU network

Paper figures, drawn from the saved results. The summary figures use the non-negativity correction.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../../"))

import numpy as np

from src import plots as P
from src.run import Experiment, Results
from src.plots import (load_tables, move_corner_label, place_captions, set_font_sizes,
                       set_partition_titles, stack_captions, thin_tick_labels)
from src.hybrid import Hybrid, HybridConfig
from src.plots import CASE_FIGURES
# After the imports: src.plots switches matplotlib to Agg.
%matplotlib inline

CASE = "osu"
OUT_DIR = "."
SWEEP_DIR = "./sweep"
FIG_DIR = "figures"
POSITIVITY = "on"

## Results

In [ ]:
sweep, rank, hybrid = load_tables(OUT_DIR, POSITIVITY)
taus = sorted(np.unique(sweep["test"]["tau_split"]))
etas = sorted(np.unique(sweep["test"]["eta"]), reverse=True)
qoi = [c.split("|", 1)[1] for c in rank["test"].dtype.names if c.startswith("l2-in-time|")]
n_eval = {s: int(np.max(sweep[s]["n_tracers"])) for s in sweep}
# tau = 0.4 is not shown.
taus = [t for t in taus if t != 0.4]
# Ranks 25-29 keep every mode.
rank = {s: r[r["residual_energy"] > 1e-15] for s, r in rank.items()}

## Errors

In [ ]:
fig_errors, axes = P.sweep_panel_splits(
    sweep["train"], rank["train"], sweep["test"], rank["test"], qoi,
    n_eval_train=n_eval["train"], n_eval_test=n_eval["test"],
    show_tols=tuple(taus), twin_tol=0.01,
    hybrid_train=hybrid["train"], hybrid_test=hybrid["test"],
    ylim=(5e-5, 4e-1), sharex=False, row_gap=0.05, figsize=(20, 11.5), legend_ncol=6, legend_y=0.95,
)
set_font_sizes(fig_errors)
# Label every other eta.
keep = {0, 2, 4, 6, 8}
thin_tick_labels(fig_errors, [axes[0][1], axes[1][1]], keep)

red, purple, orange = "#d62728", "#4a148c", "#d9601b"
for row in (0, 1):
    axes[row][0].set_ylim(1e-5, 3e2)
    axes[row][1].set_ylim(3e-5, 1e-1)
    move_corner_label(axes[row][1])
    # Drop the failure labels given as counts ("2/4096"), i.e. below 0.05%.
    for t in list(axes[row][1].texts):
        if "/" in t.get_text():
            t.remove()
# Failure label positions: (dx, dy, ha, va[, y]), offsets in points, y in data units.
up = (0, 0, "center", "bottom")
edge = (-8, 0, "left", "bottom")
place_captions(axes, {
    (0, 0): {("66%", red, 2.9e-10): (-4, -10, "right", "top"),
             ("67%", red, 2.3e-11): (0, 10, "center", "bottom"),
             ("65%", red, 3.7e-12): (-8, -10, "right", "top"),
             ("10%", red, 2.0e-13): (0, -10, "center", "top")},
    (1, 0): {("65%", red, 2.9e-10): (-4, -10, "right", "top"),
             ("65%", red, 2.3e-11): (0, 10, "center", "bottom"),
             ("61%", red, 3.7e-12): (-8, -10, "right", "top"),
             ("8%", red, 2.0e-13): (0, -10, "center", "top")},
    (0, 1): {("0.1%", orange, 1e-5): (*edge, 2.6e-2),
             ("5%", purple, 1e-7): (*up, 1.2e-2)},
    (1, 1): {("0.1%", orange, 1e-6): (*up, 1.3e-2),
             ("3%", purple, 1e-7): (8, 0, "left", "bottom", 1.0e-2)},
})

## Clusters, ranks and FOM use

In [ ]:
fig_partition, axes_p = P.sweep_partition_panel(
    sweep["test"], SWEEP_DIR, show_tols=tuple(taus), positivity=POSITIVITY, eval_split="test",
    max_eta_exp=14, bar_headroom=1.1,
)
set_font_sizes(fig_partition)
set_partition_titles(fig_partition, axes_p)
for ax in axes_p:
    thin_tick_labels(fig_partition, [ax], keep)

## Hybrid error distributions

In [ ]:
fig_violin, axes_v = P.violin_panel(
    SWEEP_DIR, sorted(taus, reverse=True), qoi, etas=etas, positivity=POSITIVITY, eval_split="test",
    prefix="hybrid_", figsize=(5 * len(taus) + 2, 5 * len(qoi)),
)
# Room above the violins for the failure labels; the label over the first violin is moved off the y axis.
for row in axes_v:
    lo, hi = row[0].get_ylim()
    row[0].set_ylim(lo, hi + 0.8)
    for ax in row:
        for t in ax.texts:
            if hasattr(t, "xy") and np.isclose(t.xy[0], 1):
                t.set_ha("left")
                t.set_position((-12, 4))

## Hybrid and non-negativity on two test tracers

Left: hybrid against AT-ROM, both without the non-negativity correction. Right: AT-ROM with and without the correction.

In [ ]:
HYBRID_TAU, HYBRID_ETA, HYBRID_TRACER = 0.01, 1e-10, 886
NONNEG_TAU, NONNEG_ETA, NONNEG_TRACER = 0.005, 1e-7, 1074

exp = Experiment(OUT_DIR, "test")
cfg, ctx = exp.cfg, exp.ctx
_, params, _ = exp.split_files


def load_case(tau, eta, i):
    """The hybrid solver, its arguments for test tracer i with the (tau, eta) model, and its FOM times and states."""
    atrom = exp.load_atrom(tau, eta)
    t, y, x = exp.trajectories([i])
    return Hybrid(atrom, cfg.replace(eta=eta), HybridConfig(positivity=False)), (params[i], x[0], ctx.dt_hydro), t[0], y[0]


solver, run, t_h, y_h = load_case(HYBRID_TAU, HYBRID_ETA, HYBRID_TRACER)
hybrid = solver.solve(*run, t_eval=t_h)
rom = solver.solve_pure(*run, t_eval=t_h, positivity=False)
print(hybrid.summary())

solver, run, t_n, y_n = load_case(NONNEG_TAU, NONNEG_ETA, NONNEG_TRACER)
rom_on = solver.solve_pure(*run, t_eval=t_n, positivity=True)
rom_off = solver.solve_pure(*run, t_eval=t_n, positivity=False)

def title(name, tau, eta):
    return rf"{name}: $\tau_\mathrm{{split}} = {tau:g}$, $\eta = 10^{{{np.log10(eta):.0f}}}$"

common = dict(qoi_indices=ctx.qoi_indices, species=ctx.net.species, labels={}, xscale="linear",
              time_scale=CASE_FIGURES[CASE]["time_scale"], time_label=CASE_FIGURES[CASE]["time_label"], time_unit="kyr")
fig_pair, _ = P.qoi_panels_pair(
    dict(result=hybrid, rom=rom, t_ref=t_h, y_ref=y_h, title=title("Hybrid", HYBRID_TAU, HYBRID_ETA), **common),
    dict(rom_on=rom_on, rom_off=rom_off, t_ref=t_n, y_ref=y_n, dt_hydro=ctx.dt_hydro,
         title=title("Non-negativity", NONNEG_TAU, NONNEG_ETA), **common))

## Save

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
fig_errors.savefig(f"{FIG_DIR}/panel_errors.pdf", dpi=200, bbox_inches="tight")
fig_partition.savefig(f"{FIG_DIR}/panel_partition.pdf", dpi=200, bbox_inches="tight")
fig_violin.savefig(f"{FIG_DIR}/violin_hybrid.pdf", bbox_inches="tight")
fig_pair.savefig(f"{FIG_DIR}/hybrid_nonnegativity_tracer{HYBRID_TRACER}_tracer{NONNEG_TRACER}.pdf", bbox_inches="tight")